# Feedback Score Boosting

New in Medha **0.5.0**.

The feedback API added in 0.4.0 accumulated `correct` / `incorrect` signals but did
nothing with them, beyond optional auto-invalidation. The counters were write-only.

From 0.5.0, positive feedback can **raise the similarity score** of an entry, so a
cached answer that people keep confirming becomes easier to retrieve:

```
trust           = feedback_correct / (feedback_correct + feedback_incorrect)
adjusted_score  = min(1.0, score × (1 + feedback_boost_factor × trust))
```

The feature is **off by default** (`feedback_boost_factor=0.0`), which reproduces 0.4.3
scoring exactly.

**Requires:** `pip install "medha-archai[fastembed]"`

In [ ]:
import pandas as pd

from medha import Medha, Settings
from medha.backends.memory import InMemoryBackend
from medha.embeddings.fastembed_adapter import FastEmbedAdapter
from medha.types import SearchStrategy

embedder = FastEmbedAdapter()
print("Embedder ready:", embedder.model_name, f"({embedder.dimension} dims)")

STORED_QUESTION = "What is the total revenue for the last quarter?"
STORED_QUERY = "SELECT SUM(amount) FROM sales WHERE quarter = 'Q4'"
SIMILAR_QUESTION = "Give me the income figures from the previous three months"


## 1. Measure the raw similarity

Rather than hard-coding a threshold and hoping the model cooperates, we measure the
actual cosine similarity between the two questions first. Everything below is derived
from that number, so this notebook behaves the same on any embedding model.

In [ ]:
settings = Settings(backend_type="memory", score_threshold_semantic=0.0)

async with Medha("boost_probe", embedder=embedder, backend=InMemoryBackend(), settings=settings) as probe:
    await probe.store(STORED_QUESTION, STORED_QUERY)
    await probe.clear_caches()
    raw = await probe.search(SIMILAR_QUESTION)

# search() reports the semantic tier's 0.9x-penalised confidence; undo it to get the
# underlying cosine similarity the threshold is compared against.
RAW_SCORE = raw.confidence / 0.9 if raw.strategy == SearchStrategy.SEMANTIC_MATCH else raw.confidence
print(f"strategy         : {raw.strategy.value}")
print(f"raw similarity   : {RAW_SCORE:.4f}")

# Put the threshold 12% above the raw score. Without a boost that is a guaranteed miss,
# and it needs a factor above 0.12 (at full trust) to clear — so the demo below can show
# both a factor that is too small and one that works.
THRESHOLD = round(RAW_SCORE * 1.12, 4)
print(f"threshold we use : {THRESHOLD}")

## 2. Before feedback — a near miss

With the threshold set just above the observed similarity, the entry is invisible: the
question is *almost* the same, but not close enough for the semantic tier.

In [ ]:
def make_settings(factor: float) -> Settings:
    return Settings(
        backend_type="memory",
        score_threshold_semantic=THRESHOLD,
        feedback_boost_factor=factor,
        l1_cache_max_size=0,   # keep every lookup on the vector path for this demo
    )


async def run_demo(factor: float, positive_feedback: int, negative_feedback: int = 0):
    """Store one entry, apply feedback, then search with a paraphrased question."""
    settings = make_settings(factor)
    async with Medha("boost_demo", embedder=embedder, backend=InMemoryBackend(), settings=settings) as m:
        await m.store(STORED_QUESTION, STORED_QUERY)
        for _ in range(positive_feedback):
            await m.feedback(STORED_QUESTION, correct=True)
        for _ in range(negative_feedback):
            await m.feedback(STORED_QUESTION, correct=False)
        await m.clear_caches()
        return await m.search(SIMILAR_QUESTION)


before = await run_demo(factor=0.0, positive_feedback=0)
print("strategy:", before.strategy.value)
print("query   :", before.generated_query or "(nothing returned)")

## 3. After feedback — the same query now hits

Five confirmations give `trust = 1.0`. With `feedback_boost_factor=0.3` the score is
multiplied by 1.3, which clears the threshold.

In [ ]:
after = await run_demo(factor=0.3, positive_feedback=5)
print("strategy:", after.strategy.value)
print("query   :", after.generated_query or "(nothing returned)")

In [ ]:
rows = []
for label, factor, pos, neg in [
    ("no feedback, no boost",      0.0, 0, 0),
    ("5 positive, boosting off",   0.0, 5, 0),
    ("5 positive, factor 0.1",     0.1, 5, 0),
    ("5 positive, factor 0.3",     0.3, 5, 0),
    ("3 positive / 2 negative",    0.3, 3, 2),
    ("5 negative only",            0.3, 0, 5),
]:
    result = await run_demo(factor, pos, neg)
    trust = pos / (pos + neg) if (pos + neg) else 0.0
    rows.append({
        "scenario": label,
        "factor": factor,
        "trust": round(trust, 2),
        "adjusted": round(min(1.0, RAW_SCORE * (1 + factor * trust)), 4),
        "threshold": THRESHOLD,
        "strategy": result.strategy.value,
        "hit": result.strategy != SearchStrategy.NO_MATCH,
    })

pd.DataFrame(rows)

Read the table top to bottom:

- **Boosting off** — feedback changes nothing, however much of it there is. This is 0.4.3
  behaviour, and it is what you get by default.
- **Factor too small** — `0.1` lifts the score but not past the threshold. The boost is a
  nudge, not an override.
- **Factor 0.3 with full trust** — clears the threshold, the entry is returned.
- **Mixed feedback** — `trust = 3/5 = 0.6` only gets 60% of the boost.
- **Negative only** — `trust = 0.0`, so the score is untouched. **Boosting never
  penalises**; to remove bad entries use `feedback_incorrect_threshold` instead.

## 4. Where the boost applies

Only where similarity ranking is actually involved:

| tier | boosted? | why |
|---|---|---|
| L1 cache | no | exact question hash — already a certainty |
| Template | no | pattern match, not a similarity score |
| Exact | no | hash-equivalent match |
| **Semantic** | **yes** | cosine similarity ranking |
| **Fuzzy** | **yes** | Levenshtein ratio ranking |

When boosting is enabled the semantic tier also retrieves candidates slightly *below*
the threshold — down to `threshold / (1 + factor)`, the lowest score that could still be
lifted over the line. Nothing below that bound is reachable no matter how good its
feedback, so recall widens by exactly the amount the boost can justify and no more.

A second effect: boosting **re-ranks**. A marginally less similar entry with a strong
track record can now outrank a closer one that nobody has confirmed.

## 5. Choosing a factor

| factor | effect |
|---|---|
| `0.0` *(default)* | disabled — identical to 0.4.3 |
| `0.1` | subtle; breaks ties between near-equal candidates |
| `0.2` – `0.3` | recommended starting range |
| `0.5`+ | aggressive; a trusted entry can absorb a large similarity gap |

Start at `0.2`, measure the hit rate before and after (see
[28_persistent_stats.ipynb](28_persistent_stats.ipynb) — persisted stats make this
comparison easy across restarts), and raise it only if the extra hits are actually
correct.

The risk of a high factor is over-retrieval: a well-rated entry starts answering
questions it should not. Feedback signal quality matters more than the factor — a
handful of confirmations from real users beats hundreds of synthetic ones.

Configure it with `Settings(feedback_boost_factor=...)` or `MEDHA_FEEDBACK_BOOST_FACTOR`.

## 6. Composing with auto-invalidation

The two feedback features work in opposite directions and complement each other:

```python
Settings(
    feedback_boost_factor=0.25,        # confirmed answers get easier to find
    feedback_incorrect_threshold=3,    # answers rejected 3 times are deleted
)
```

Positive feedback promotes good entries; negative feedback removes bad ones once it
crosses the threshold. Between the two the cache converges on the answers your users
actually accept, without anyone curating it by hand.

In [ ]:
settings = Settings(
    backend_type="memory",
    score_threshold_semantic=THRESHOLD,
    feedback_boost_factor=0.25,
    feedback_incorrect_threshold=3,
    l1_cache_max_size=0,
)

async with Medha("combined", embedder=embedder, backend=InMemoryBackend(), settings=settings) as m:
    await m.store(STORED_QUESTION, STORED_QUERY)

    # Three rejections trip the auto-invalidation threshold.
    for _ in range(3):
        await m.feedback(STORED_QUESTION, correct=False)
    await m.clear_caches()

    result = await m.search(STORED_QUESTION)
    print("after 3 rejections:", result.strategy.value, "— entry removed from the cache")